# Teacher Screening with Early Stopping

**Goal.** Train every knowledge-distillation (KD) teacher candidate with one identical recipe plus early stopping, then export — for each model — training/validation curves, a test-set confusion matrix, ROC curves, test metrics and training time, so the best teacher for the EfficientNetB0 student can be chosen.

### Training recipe
Identical to the repo's `6.1 * Base.ipynb` baselines. **The only addition is early stopping.**

| Setting | Value |
|---|---|
| Data | `SplittedDataset/{train,val,test}` — 1614 / 229 / 466 knee X-rays, KL grades 0–4 |
| Preprocessing | `bphe_rgb`: histogram-equalise the luma (Y) channel in YCrCb space, then the backbone's own `preprocess_input` |
| Resize | straight to the backbone's native input size (`flow_from_directory`, RGB) |
| Augmentation (train only) | rotation 30°, width/height shift 0.2, shear 0.2, zoom 0.2, horizontal flip, brightness 0.8–1.2, fill `nearest` |
| Backbone | ImageNet weights, **all backbone layers frozen** |
| Head | pooled features → `Dense(5, softmax)` |
| Optimiser / loss | Adam, learning rate 1e-4 · categorical cross-entropy |
| Batch size / max epochs | 32 / 100 |
| **Early stopping (new)** | `monitor='val_loss'`, `patience=10`, `restore_best_weights=True` — the test set is evaluated with the **best-epoch** weights |

### Why Adam with learning rate 1e-4
Grid search from notebook 2 (EfficientNetB5 + custom head; validation accuracy):

| LR | Dropout | RMSprop 16 | RMSprop 32 | Adam 16 | Adam 32 | SGD 16 | SGD 32 |
|---|---|---|---|---|---|---|---|
| 0.0001 | 0.3 | 0.9039 | **0.9083** | **0.9083** | 0.8908 | 0.4847 | 0.3668 |
| 0.0001 | 0.5 | 0.8996 | 0.8952 | 0.8952 | 0.9039 | 0.5109 | 0.3668 |
| 0.001 | 0.3 | 0.8908 | 0.8734 | 0.8646 | 0.8952 | 0.8210 | 0.7642 |
| 0.001 | 0.5 | 0.8515 | 0.8821 | 0.8690 | 0.8690 | 0.8472 | 0.7293 |
| 0.0003 | 0.3 | 0.9039 | 0.8908 | 0.8952 | 0.8865 | 0.6769 | 0.6201 |
| 0.0003 | 0.5 | 0.8865 | 0.8908 | 0.8908 | 0.8908 | 0.6856 | 0.6157 |

Adam (batch 16) and RMSprop (batch 32) tie for best at learning rate 1e-4. The 6.1 baselines already use Adam with learning rate 1e-4; the batch size stays at 32 so the recipe is identical to 6.1. Dropout does not apply here, because the frozen-backbone head has no dropout layer.

### Models
| Section | Model | Framework | Input | Head |
|---|---|---|---|---|
| 5.1–5.6 | EfficientNetB0 · B1 · B2 · B3 · B4 · B5 | Keras | 224 · 240 · 260 · 300 · 380 · 456 | ImageNet top kept up to `top_dropout`, then `Dense(5)` (as in `6.1. EfficientNet Base`) |
| 5.7–5.8 | ResNet50 · ResNet101 | Keras | 224 | `GlobalAveragePooling2D` → `Dense(5)` |
| 5.9–5.10 | DenseNet121 · DenseNet201 | Keras | 224 | `GlobalAveragePooling2D` → `Dense(5)` |
| 5.11–5.12 | InceptionV3 · InceptionResNetV2 | Keras | 299 | `GlobalAveragePooling2D` → `Dense(5)` |
| 5.13 | MobileNetV2 | Keras | 224 | `GlobalAveragePooling2D` → `Dense(5)` |
| 6.1–6.2 | Twins-SVT-S · PVTv2-B2 | PyTorch (timm) | 224 | token/spatial average pooling → `Linear(5)` |

EfficientNetB0 and MobileNetV2 are baselines (not teacher candidates). **Custom-ENB5** (notebook 3) is not retrained; its published numbers appear as a reference row in the summary (section 7).

### Outputs
Everything is written to `results/` next to this notebook (i.e. `teacher/results/`):
```
results/<ModelName>/
    figures/training_curves.{pdf,png}   figures/confusion_matrix.{pdf,png}   figures/roc_curve.{pdf,png}
    metrics.json                (all test metrics, timing, epochs, settings; written last)
    history.csv                 (per-epoch loss / accuracy / val_loss / val_accuracy / epoch time)
    classification_report.txt   classification_report.csv   confusion_matrix.csv
    test_predictions.csv        (file, true grade, predicted grade, class probabilities)
    best.keras | head_best.pt   (best-epoch weights, reusable as a KD teacher)
results/_summary/
    summary.csv   advisor_table.csv   figures/*.{pdf,png}
```
Figures are saved as vector PDF plus 600-dpi PNG.

### How to run
1. Open this notebook with the TensorFlow GPU (WSL) kernel (`tf_gpu_env`: TensorFlow 2.19 / Keras 3 + PyTorch).
2. Check `DATA_DIR` in section 1.1.
3. *Run All.* Finished models are skipped on a re-run (`SKIP_EXISTING`), so a crash or kernel restart only loses the model in progress.
4. If the PyTorch models run out of GPU memory after all the Keras models, restart the kernel, set `MODELS = ['Twins-SVT-S', 'PVTv2-B2']` in section 1.1 and run all again: the Keras results are loaded from disk.

## 1. Environment & Configuration
### 1.1 Settings
Edit the values here. Each one can also be overridden with an `OSTEO_*` environment variable (used for quick test runs).

In [ ]:
import os

# Must be set before TensorFlow starts: let TensorFlow and PyTorch share the GPU, hide TF info logs
os.environ.setdefault("TF_FORCE_GPU_ALLOW_GROWTH", "true")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")


def _env(key, default, cast=str):
    value = os.environ.get(key)
    return default if value in (None, "") else cast(value)


CFG = dict(
    DATA_DIR=_env("OSTEO_DATA_DIR", "/mnt/d/FA019/new/SplittedDataset"),
    OUTPUT_DIR=_env("OSTEO_OUTPUT_DIR", "results"),      # -> teacher/results/<ModelName>/
    MODELS=_env("OSTEO_MODELS", "all"),                  # "all", a list, or "Name1,Name2"
    MAX_EPOCHS=_env("OSTEO_MAX_EPOCHS", 100, int),
    PATIENCE=_env("OSTEO_PATIENCE", 10, int),
    SKIP_EXISTING=_env("OSTEO_SKIP_EXISTING", 1, int) == 1,
    SAVE_WEIGHTS=_env("OSTEO_SAVE_WEIGHTS", 1, int) == 1,
    SEED=_env("OSTEO_SEED", 42, int),
    BATCH_SIZE=32,
    LR=1e-4,
    MONITOR="val_loss",
)

KERAS_ORDER = ["EfficientNetB0", "EfficientNetB1", "EfficientNetB2", "EfficientNetB3", "EfficientNetB4",
               "EfficientNetB5", "ResNet50", "ResNet101", "DenseNet121", "DenseNet201", "InceptionV3",
               "InceptionResNetV2", "MobileNetV2"]
TIMM_ORDER = ["Twins-SVT-S", "PVTv2-B2"]
ALL_MODELS = KERAS_ORDER + TIMM_ORDER

_models = CFG["MODELS"]
if isinstance(_models, str):
    _models = ALL_MODELS if _models.strip().lower() == "all" else [m.strip() for m in _models.split(",") if m.strip()]
_unknown = sorted(set(_models) - set(ALL_MODELS))
assert not _unknown, f"Unknown model names: {_unknown}. Choose from {ALL_MODELS}"
SELECTED = [m for m in ALL_MODELS if m in _models]
print("Models to run:", SELECTED)
CFG

### 1.2 Packages
Installs `timm` (Twins / PVTv2) and `SciencePlots` (figure style) only if they are missing.

In [ ]:
import importlib.util
import subprocess
import sys

for pip_name, module in [("timm", "timm"), ("SciencePlots", "scienceplots")]:
    if importlib.util.find_spec(module) is None:
        print(f"Installing {pip_name} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])

### 1.3 Imports
PyTorch is imported before TensorFlow, as in the original notebooks, and TensorFlow's GPU memory growth is switched on before any GPU work.

In [ ]:
import gc
import json
import logging
import platform
import time
import warnings
from datetime import datetime
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import PIL
from PIL import Image
from IPython.display import display

import torch
import torch.nn as nn
import timm

import tensorflow as tf

for _gpu in tf.config.list_physical_devices("GPU"):
    try:
        tf.config.experimental.set_memory_growth(_gpu, True)
    except RuntimeError as err:  # already initialised
        print("Memory growth not set:", err)

import keras
from tensorflow.keras.applications import (
    DenseNet121, DenseNet201, EfficientNetB0, EfficientNetB1, EfficientNetB2, EfficientNetB3, EfficientNetB4,
    EfficientNetB5, InceptionResNetV2, InceptionV3, MobileNetV2, ResNet50, ResNet101,
    densenet, efficientnet, inception_resnet_v2, inception_v3, mobilenet_v2, resnet50,
)
from tensorflow.keras.callbacks import Callback, EarlyStopping
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator

from sklearn.metrics import (accuracy_score, auc, classification_report, confusion_matrix, f1_score,
                             precision_score, recall_score, roc_auc_score, roc_curve)
from sklearn.preprocessing import label_binarize

import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import MaxNLocator

warnings.filterwarnings("ignore", message=".*PyDataset.*")
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)

### 1.4 Versions & GPU check

In [ ]:
assert keras.__version__.startswith("3"), f"Expected Keras 3 (TensorFlow 2.16+), found {keras.__version__}"

VERSIONS = {"python": platform.python_version(), "tensorflow": tf.__version__, "keras": keras.__version__,
            "torch": torch.__version__, "timm": timm.__version__, "numpy": np.__version__,
            "opencv": cv2.__version__, "pillow": PIL.__version__}
display(pd.Series(VERSIONS, name="version").to_frame())

TF_GPUS = tf.config.list_physical_devices("GPU")
with tf.device("/GPU:0" if TF_GPUS else "/CPU:0"):
    tf.reduce_sum(tf.matmul(tf.random.normal((256, 256)), tf.random.normal((256, 256)))).numpy()
print("TensorFlow GPU:", [g.name for g in TF_GPUS] or "none - running on CPU")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.randn(8, 3, 32, 32, device=DEVICE).sum().item()
print("PyTorch device:", torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "cpu")

### 1.5 Paths

In [ ]:
DATA_DIR = Path(CFG["DATA_DIR"])
TRAIN_DIR, VAL_DIR, TEST_DIR = (DATA_DIR / split for split in ("train", "val", "test"))
for folder in (TRAIN_DIR, VAL_DIR, TEST_DIR):
    assert folder.is_dir(), f"Missing data folder: {folder}"

OUT_DIR = Path(CFG["OUTPUT_DIR"])
SUMMARY_DIR = OUT_DIR / "_summary"
SUMMARY_DIR.mkdir(parents=True, exist_ok=True)
print("Data   :", DATA_DIR.resolve())
print("Results:", OUT_DIR.resolve())

## 2. Figure Style
Publication style used for every figure: SciencePlots (`science` + `ieee` + `no-latex`, with a plain-matplotlib fallback), bold serif text, thick axes and colour-blind-safe Okabe–Ito colours. Each figure is saved as vector PDF plus 600-dpi PNG and shown inline at screen resolution.

In [ ]:
%matplotlib inline
%config InlineBackend.figure_formats = ["png"]

try:
    import scienceplots  # noqa: F401  (registers the styles)
    plt.style.use(["science", "ieee", "no-latex"])
    SCIENCEPLOTS = True
except Exception:
    SCIENCEPLOTS = False

plt.rcParams.update({
    "figure.dpi": 100, "savefig.dpi": 600, "savefig.bbox": "tight", "savefig.facecolor": "white",
    "font.family": "serif", "font.serif": ["Times New Roman", "DejaVu Serif"], "mathtext.fontset": "stix",
    "font.size": 16, "font.weight": "bold",
    "axes.labelsize": 18, "axes.labelweight": "bold", "axes.titlesize": 18, "axes.titleweight": "bold",
    "xtick.labelsize": 14, "ytick.labelsize": 14,
    "legend.fontsize": 13, "legend.title_fontsize": 14, "figure.titlesize": 20,
    "axes.linewidth": 1.5, "grid.linewidth": 1.0, "lines.linewidth": 2.4, "patch.linewidth": 1.2,
    "xtick.major.width": 1.5, "ytick.major.width": 1.5, "xtick.major.size": 6, "ytick.major.size": 6,
    "legend.frameon": True, "legend.framealpha": 0.95, "legend.edgecolor": "black", "legend.fancybox": True,
    "pdf.fonttype": 42, "ps.fonttype": 42,
})

OKABE_ITO = ["#0072B2", "#D55E00", "#009E73", "#CC79A7", "#E69F00", "#56B4E9", "#F0E442", "#000000"]
C_TRAIN, C_VAL, C_REF, C_CHANCE = "#0072B2", "#D55E00", "#222222", "#7F7F7F"
CLASS_COLORS = OKABE_ITO[:5]
FAMILY_COLORS = {"EfficientNet": "#0072B2", "ResNet": "#D55E00", "DenseNet": "#009E73",
                 "Inception": "#CC79A7", "MobileNet": "#E69F00", "Transformer": "#56B4E9"}


def save_fig(fig, stem, save=True):
    """Save `fig` as <stem>.pdf and <stem>.png (600 dpi), show it inline, then close it."""
    if save:
        stem = Path(stem)
        stem.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(f"{stem}.pdf")
        fig.savefig(f"{stem}.png", dpi=600)
    plt.show()
    plt.close(fig)


print("SciencePlots style:", "on" if SCIENCEPLOTS else "not available - using rcParams only")

## 3. Data Audit
Image counts per split and grade, and the image formats present. Keras loads every file as 8-bit RGB (`color_mode='rgb'`), exactly as in the original notebooks.

In [ ]:
CLASS_NAMES = sorted(p.name for p in TRAIN_DIR.iterdir() if p.is_dir())
assert CLASS_NAMES == ["0", "1", "2", "3", "4"], f"Unexpected class folders: {CLASS_NAMES}"
NUM_CLASSES = len(CLASS_NAMES)
SPLIT_DIRS = {"train": TRAIN_DIR, "val": VAL_DIR, "test": TEST_DIR}

counts = pd.DataFrame({split: {c: sum(f.is_file() for f in (d / c).iterdir()) for c in CLASS_NAMES}
                       for split, d in SPLIT_DIRS.items()})
counts.index.name = "KL grade"
counts.loc["total"] = counts.sum()
display(counts)

rows = []
for split, d in SPLIT_DIRS.items():
    for c in CLASS_NAMES:
        for f in (d / c).iterdir():
            with Image.open(f) as im:
                rows.append((split, c, im.mode, f"{im.size[0]}x{im.size[1]}"))
census = pd.DataFrame(rows, columns=["split", "KL grade", "mode", "size"])
display(census.groupby(["mode", "size", "KL grade"]).size().unstack("KL grade", fill_value=0))

## 4. Shared Helpers
### 4.1 Preprocessing & data generators
`bphe_rgb` and the three generators are the 6.1 code, unchanged. The only difference is a fixed `seed` for the training shuffle, for reproducibility.

In [ ]:
AUGMENTATION = dict(rotation_range=30, width_shift_range=0.2, height_shift_range=0.2, shear_range=0.2,
                    zoom_range=0.2, horizontal_flip=True, brightness_range=[0.8, 1.2], fill_mode="nearest")


def bphe_uint8(img):
    """BPHE step of the original `bphe_rgb`: equalise the Y channel (YCrCb), return uint8 RGB."""
    img = img.astype(np.uint8) if img.max() > 1 else (img * 255).astype(np.uint8)
    y, cr, cb = cv2.split(cv2.cvtColor(img, cv2.COLOR_RGB2YCrCb))
    y_eq = cv2.equalizeHist(y)
    return cv2.cvtColor(cv2.merge((y_eq, cr, cb)), cv2.COLOR_YCrCb2RGB)


def make_bphe(preprocess_input):
    """The original `bphe_rgb`: BPHE, then the backbone's own `preprocess_input`."""
    def bphe_rgb(img):
        return preprocess_input(bphe_uint8(img).astype(np.float32))
    return bphe_rgb


def assert_class_order(*generators):
    expected = {c: i for i, c in enumerate(CLASS_NAMES)}
    for g in generators:
        assert g.class_indices == expected, f"Class order mismatch: {g.class_indices}"


def make_generators(img_size, preprocess_fn, seed=None):
    """Train (augmented, shuffled), val (shuffled, as in 6.1) and test (not shuffled) generators."""
    size, bs = (img_size, img_size), CFG["BATCH_SIZE"]
    train_gen = ImageDataGenerator(preprocessing_function=preprocess_fn, **AUGMENTATION).flow_from_directory(
        str(TRAIN_DIR), target_size=size, batch_size=bs, class_mode="categorical", seed=seed)
    val_gen = ImageDataGenerator(preprocessing_function=preprocess_fn).flow_from_directory(
        str(VAL_DIR), target_size=size, batch_size=bs, class_mode="categorical", seed=seed)
    test_gen = ImageDataGenerator(preprocessing_function=preprocess_fn).flow_from_directory(
        str(TEST_DIR), target_size=size, batch_size=bs, class_mode="categorical", shuffle=False)
    assert_class_order(train_gen, val_gen, test_gen)
    return train_gen, val_gen, test_gen


def make_eval_generator(folder, img_size, preprocess_fn):
    """Unshuffled, un-augmented generator (used to re-check the restored best weights)."""
    return ImageDataGenerator(preprocessing_function=preprocess_fn).flow_from_directory(
        str(folder), target_size=(img_size, img_size), batch_size=CFG["BATCH_SIZE"],
        class_mode="categorical", shuffle=False)

### 4.2 Timing
`EpochTimer` records the wall-clock time of every epoch (training + validation). The total training time is measured around `fit`, so model building and weight downloads are excluded.

In [ ]:
class EpochTimer(Callback):
    def on_train_begin(self, logs=None):
        self.epoch_times = []

    def on_epoch_begin(self, epoch, logs=None):
        self._start = time.perf_counter()

    def on_epoch_end(self, epoch, logs=None):
        self.epoch_times.append(time.perf_counter() - self._start)


def history_info(hist, best_idx, fit_time_s, stopped_early):
    """Epoch / timing summary from a history table (columns: accuracy, loss, val_accuracy, val_loss, epoch_time_s)."""
    times = hist["epoch_time_s"].to_numpy()
    return dict(
        epochs_run=len(hist), best_epoch=best_idx + 1, stopped_early=bool(stopped_early),
        best_val_loss=float(hist["val_loss"].iloc[best_idx]),
        val_accuracy_at_best=float(hist["val_accuracy"].iloc[best_idx]),
        train_accuracy_at_best=float(hist["accuracy"].iloc[best_idx]),
        final_val_accuracy=float(hist["val_accuracy"].iloc[-1]),
        max_val_accuracy=float(hist["val_accuracy"].max()),
        max_val_accuracy_epoch=int(hist["val_accuracy"].to_numpy().argmax()) + 1,
        fit_time_s=float(fit_time_s), fit_time_min=float(fit_time_s) / 60,
        time_to_best_s=float(times[: best_idx + 1].sum()),
        epoch1_s=float(times[0]),
        median_epoch_s=float(np.median(times[1:])) if len(times) > 1 else float(times[0]),
    )

### 4.3 Test metrics
Accuracy; precision, recall and F1 (weighted and macro); the per-grade classification report; the confusion matrix; one-vs-rest ROC AUC per grade, plus macro, weighted and micro averages.

In [ ]:
def compute_metrics(y_true, prob):
    labels = list(range(NUM_CLASSES))
    y_pred = prob.argmax(axis=1)
    y_bin = label_binarize(y_true, classes=labels)
    auc_per_class = {}
    for i, c in enumerate(CLASS_NAMES):
        if y_bin[:, i].min() == y_bin[:, i].max():
            auc_per_class[c] = float("nan")
        else:
            fpr, tpr, _ = roc_curve(y_bin[:, i], prob[:, i])
            auc_per_class[c] = float(auc(fpr, tpr))
    fpr_micro, tpr_micro, _ = roc_curve(y_bin.ravel(), prob.ravel())
    kw = dict(zero_division=0)
    scalars = dict(
        test_accuracy=accuracy_score(y_true, y_pred),
        test_precision_weighted=precision_score(y_true, y_pred, average="weighted", **kw),
        test_recall_weighted=recall_score(y_true, y_pred, average="weighted", **kw),
        test_f1_weighted=f1_score(y_true, y_pred, average="weighted", **kw),
        test_precision_macro=precision_score(y_true, y_pred, average="macro", **kw),
        test_recall_macro=recall_score(y_true, y_pred, average="macro", **kw),
        test_f1_macro=f1_score(y_true, y_pred, average="macro", **kw),
        test_auc_macro=float(np.nanmean(list(auc_per_class.values()))),
        test_auc_weighted=roc_auc_score(y_bin, prob, average="weighted"),
        test_auc_micro=auc(fpr_micro, tpr_micro),
    )
    return dict(
        scalars={k: float(v) for k, v in scalars.items()},
        auc_per_class=auc_per_class,
        confusion_matrix=confusion_matrix(y_true, y_pred, labels=labels),
        report_text=classification_report(y_true, y_pred, labels=labels, target_names=CLASS_NAMES, digits=4, **kw),
        report_dict=classification_report(y_true, y_pred, labels=labels, target_names=CLASS_NAMES,
                                          output_dict=True, **kw),
    )

### 4.4 Plots
Three figures per model: training vs validation curves (the dotted line marks the best epoch, whose weights are restored), the test confusion matrix (counts and row percentages), and one-vs-rest ROC curves.

In [ ]:
def plot_curves(name, hist, best_epoch, stem, save=True):
    epochs = np.arange(1, len(hist) + 1)
    marker = "o" if len(hist) <= 30 else None
    fig, axes = plt.subplots(1, 2, figsize=(14.8, 5.6))
    for ax, key, label in ((axes[0], "accuracy", "Accuracy"), (axes[1], "loss", "Loss")):
        ax.plot(epochs, hist[key], color=C_TRAIN, ls="-", lw=2.6, marker=marker, ms=5, label="Training")
        ax.plot(epochs, hist[f"val_{key}"], color=C_VAL, ls="-", lw=2.6, marker=marker, ms=5, label="Validation")
        ax.axvline(best_epoch, color=C_REF, ls=":", lw=2.2, label=f"Best epoch ({best_epoch})")
        ax.xaxis.set_major_locator(MaxNLocator(integer=True))
        ax.set_xlabel("Epoch")
        ax.set_ylabel(label)
        ax.set_title(label)
        ax.grid(True, alpha=0.3, linestyle="--")
        ax.set_axisbelow(True)
        ax.legend(loc="best")
    fig.suptitle(f"{name}: training and validation", fontweight="bold", y=1.04)
    save_fig(fig, stem, save)


def plot_confusion(name, cm, stem, save=True):
    cm = np.asarray(cm)
    pct = cm / np.clip(cm.sum(axis=1, keepdims=True), 1, None) * 100
    annot = np.array([[f"{n}\n({p:.1f}%)" for n, p in zip(row, prow)] for row, prow in zip(cm, pct)])
    fig, ax = plt.subplots(figsize=(8.2, 7.0))
    sns.heatmap(cm, annot=annot, fmt="", cmap="Blues", square=True, linewidths=0.8, linecolor="white",
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, annot_kws={"size": 13, "weight": "bold"},
                cbar_kws={"label": "Images"}, ax=ax)
    ax.minorticks_off()
    ax.tick_params(top=False, right=False)
    ax.set_xlabel("Predicted KL grade")
    ax.set_ylabel("True KL grade")
    ax.set_title(f"{name}: confusion matrix (test)")
    save_fig(fig, stem, save)


def plot_roc(name, y_true, prob, res, stem, save=True):
    y_bin = label_binarize(y_true, classes=list(range(NUM_CLASSES)))
    fig, ax = plt.subplots(figsize=(7.6, 7.0))
    for i, c in enumerate(CLASS_NAMES):
        if y_bin[:, i].min() == y_bin[:, i].max():
            continue
        fpr, tpr, _ = roc_curve(y_bin[:, i], prob[:, i])
        ax.plot(fpr, tpr, color=CLASS_COLORS[i], ls="-", lw=2.4, label=f"KL {c} (AUC = {res['auc_per_class'][c]:.3f})")
    fpr, tpr, _ = roc_curve(y_bin.ravel(), prob.ravel())
    ax.plot(fpr, tpr, color=C_REF, ls=":", lw=2.6, label=f"Micro-average (AUC = {res['test_auc_micro']:.3f})")
    ax.plot([0, 1], [0, 1], color=C_CHANCE, ls="--", lw=1.8, label="Chance")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1.02)
    ax.set_aspect("equal")
    ax.set_xlabel("False positive rate")
    ax.set_ylabel("True positive rate")
    ax.set_title(f"{name}: ROC (test)\nmacro AUC = {res['test_auc_macro']:.3f}")
    ax.grid(True, alpha=0.3, linestyle="--")
    ax.set_axisbelow(True)
    ax.legend(loc="lower right", fontsize=11)
    save_fig(fig, stem, save)


def show_model_figures(name, hist, y_true, prob, res, save=True):
    fig_dir = OUT_DIR / name / "figures"
    plot_curves(name, hist, res["best_epoch"], fig_dir / "training_curves", save)
    plot_confusion(name, res["confusion_matrix"], fig_dir / "confusion_matrix", save)
    plot_roc(name, y_true, prob, res, fig_dir / "roc_curve", save)

### 4.5 Saving, resuming and summaries
`finalize_run` writes every numeric result and figure for one model; `metrics.json` is written last and marks the model as finished. `run_or_load` skips finished models and re-displays their saved results.

In [ ]:
def _json_default(obj):
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.floating):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, Path):
        return str(obj)
    raise TypeError(f"Not JSON serialisable: {type(obj)}")


def write_json_atomic(path, data):
    tmp = Path(f"{path}.tmp")
    tmp.write_text(json.dumps(data, indent=2, default=_json_default))
    os.replace(tmp, path)


def print_result(name, res):
    print(f"\n{'=' * 70}\n{name}: test results (best epoch {res['best_epoch']} of {res['epochs_run']}"
          f"{', early-stopped' if res['stopped_early'] else ''})\n{'=' * 70}")
    print(f"Test accuracy          : {res['test_accuracy']:.4f}   (test loss {res['test_loss']:.4f})")
    print(f"Precision  weighted    : {res['test_precision_weighted']:.4f}   macro: {res['test_precision_macro']:.4f}")
    print(f"Recall     weighted    : {res['test_recall_weighted']:.4f}   macro: {res['test_recall_macro']:.4f}")
    print(f"F1 score   weighted    : {res['test_f1_weighted']:.4f}   macro: {res['test_f1_macro']:.4f}")
    print(f"ROC AUC    macro       : {res['test_auc_macro']:.4f}   micro: {res['test_auc_micro']:.4f}")
    print(f"Val accuracy (best ep.): {res['val_accuracy_at_best']:.4f}   best val loss: {res['best_val_loss']:.4f}")
    print(f"Training time          : {res['fit_time_min']:.1f} min  (epoch 1 {res['epoch1_s']:.0f} s, "
          f"median epoch {res['median_epoch_s']:.1f} s, time to best {res['time_to_best_s'] / 60:.1f} min)")
    report = OUT_DIR / name / "classification_report.txt"
    if report.exists():
        print("\nClassification report (test):\n" + report.read_text())


def finalize_run(name, info, hist, y_true, prob, filenames):
    """Compute test metrics; write CSV / JSON / figures for one model; return its result dict."""
    out = OUT_DIR / name
    out.mkdir(parents=True, exist_ok=True)
    assert np.allclose(prob.sum(axis=1), 1.0, atol=1e-3), "Class probabilities do not sum to 1"
    m = compute_metrics(y_true, prob)

    hist.to_csv(out / "history.csv", index=False)
    pred = pd.DataFrame({"file": filenames, "y_true": y_true, "y_pred": prob.argmax(axis=1)})
    for i, c in enumerate(CLASS_NAMES):
        pred[f"prob_{c}"] = prob[:, i]
    pred.to_csv(out / "test_predictions.csv", index=False)
    (out / "classification_report.txt").write_text(m["report_text"])
    pd.DataFrame(m["report_dict"]).T.to_csv(out / "classification_report.csv")
    pd.DataFrame(m["confusion_matrix"], index=[f"true_{c}" for c in CLASS_NAMES],
                 columns=[f"pred_{c}" for c in CLASS_NAMES]).to_csv(out / "confusion_matrix.csv")

    res = {**info, **m["scalars"], "auc_per_class": m["auc_per_class"],
           "confusion_matrix": m["confusion_matrix"].tolist(),
           "finished_at": datetime.now().isoformat(timespec="seconds")}
    show_model_figures(name, hist, y_true, prob, res, save=True)
    write_json_atomic(out / "metrics.json", res)
    print_result(name, res)
    return res


def load_result(name, show=True):
    out = OUT_DIR / name
    res = json.loads((out / "metrics.json").read_text())
    if show:
        hist = pd.read_csv(out / "history.csv")
        pred = pd.read_csv(out / "test_predictions.csv")
        prob = pred[[f"prob_{c}" for c in CLASS_NAMES]].to_numpy()
        show_model_figures(name, hist, pred["y_true"].to_numpy(), prob, res, save=False)
        print_result(name, res)
    return res


def cleanup_backend():
    keras.backend.clear_session()
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


RESULTS = {}


def run_or_load(name, runner):
    if name not in SELECTED:
        print(f"{name}: not in MODELS - skipped")
        return None
    if CFG["SKIP_EXISTING"] and (OUT_DIR / name / "metrics.json").exists():
        print(f"{name}: finished results found in {OUT_DIR / name} - loading instead of retraining")
        RESULTS[name] = load_result(name)
    else:
        RESULTS[name] = runner(name)
    return RESULTS[name]


def common_info(name, framework, family, input_size, preprocess, head, params_total, params_trainable,
                n_train, n_val, n_test):
    return dict(model=name, framework=framework, family=family, input_size=input_size, preprocess=preprocess,
                head=head, recipe="6.1 frozen-backbone baseline + EarlyStopping", optimizer="Adam",
                lr=CFG["LR"], batch_size=CFG["BATCH_SIZE"], max_epochs=CFG["MAX_EPOCHS"],
                patience=CFG["PATIENCE"], monitor=CFG["MONITOR"], seed=CFG["SEED"],
                params_total=int(params_total), params_trainable=int(params_trainable),
                n_train=int(n_train), n_val=int(n_val), n_test=int(n_test), versions=VERSIONS,
                data_dir=str(DATA_DIR))

## 5. Keras Models
### 5.0 Model registry, builder and training loop
The builders reproduce the 6.1 notebooks exactly:
- **EfficientNet** (as in `6.1. EfficientNet Base`): `include_top=True` at the native resolution, cut at `layers[-2]` (the built-in average pooling + `top_dropout`), then a new `Dense(5)`.
- **All other backbones**: `include_top=False` → `GlobalAveragePooling2D` → `Dense(5)`.

Every backbone layer is frozen, so only the final `Dense` layer trains.

In [ ]:
KERAS_MODELS = {
    "EfficientNetB0": dict(ctor=EfficientNetB0, size=224, pre=efficientnet.preprocess_input, head="top", family="EfficientNet"),
    "EfficientNetB1": dict(ctor=EfficientNetB1, size=240, pre=efficientnet.preprocess_input, head="top", family="EfficientNet"),
    "EfficientNetB2": dict(ctor=EfficientNetB2, size=260, pre=efficientnet.preprocess_input, head="top", family="EfficientNet"),
    "EfficientNetB3": dict(ctor=EfficientNetB3, size=300, pre=efficientnet.preprocess_input, head="top", family="EfficientNet"),
    "EfficientNetB4": dict(ctor=EfficientNetB4, size=380, pre=efficientnet.preprocess_input, head="top", family="EfficientNet"),
    "EfficientNetB5": dict(ctor=EfficientNetB5, size=456, pre=efficientnet.preprocess_input, head="top", family="EfficientNet"),
    "ResNet50": dict(ctor=ResNet50, size=224, pre=resnet50.preprocess_input, head="gap", family="ResNet"),
    "ResNet101": dict(ctor=ResNet101, size=224, pre=resnet50.preprocess_input, head="gap", family="ResNet"),
    "DenseNet121": dict(ctor=DenseNet121, size=224, pre=densenet.preprocess_input, head="gap", family="DenseNet"),
    "DenseNet201": dict(ctor=DenseNet201, size=224, pre=densenet.preprocess_input, head="gap", family="DenseNet"),
    "InceptionV3": dict(ctor=InceptionV3, size=299, pre=inception_v3.preprocess_input, head="gap", family="Inception"),
    "InceptionResNetV2": dict(ctor=InceptionResNetV2, size=299, pre=inception_resnet_v2.preprocess_input, head="gap", family="Inception"),
    "MobileNetV2": dict(ctor=MobileNetV2, size=224, pre=mobilenet_v2.preprocess_input, head="gap", family="MobileNet"),
}
assert list(KERAS_MODELS) == KERAS_ORDER


def build_keras_model(name):
    spec = KERAS_MODELS[name]
    shape = (spec["size"], spec["size"], 3)
    if spec["head"] == "top":   # 6.1 EfficientNet recipe: keep the ImageNet top, replace only its 1000-way Dense
        base = spec["ctor"](weights="imagenet", include_top=True, input_shape=shape)
        assert base.layers[-2].name == "top_dropout", base.layers[-2].name
        features = base.layers[-2].output
    else:                       # 6.1 recipe for the other backbones
        base = spec["ctor"](weights="imagenet", include_top=False, input_shape=shape)
        features = GlobalAveragePooling2D()(base.output)
    out = Dense(NUM_CLASSES, activation="softmax", name="pred")(features)
    model = Model(inputs=base.input, outputs=out)
    for layer in base.layers:   # freeze the feature extractor
        layer.trainable = False
    model.compile(optimizer=Adam(learning_rate=CFG["LR"]), loss="categorical_crossentropy", metrics=["accuracy"])

    feat_dim = int(features.shape[-1])
    n_trainable = int(sum(np.prod(w.shape) for w in model.trainable_weights))
    assert n_trainable == feat_dim * NUM_CLASSES + NUM_CLASSES, f"Unexpected trainable params: {n_trainable}"
    return model, feat_dim


def run_keras_model(name):
    spec = KERAS_MODELS[name]
    out = OUT_DIR / name
    out.mkdir(parents=True, exist_ok=True)
    keras.utils.set_random_seed(CFG["SEED"])

    preprocess = make_bphe(spec["pre"])
    train_gen, val_gen, test_gen = make_generators(spec["size"], preprocess, seed=CFG["SEED"])
    model, feat_dim = build_keras_model(name)
    print(f"{name}: input {spec['size']}x{spec['size']}, {feat_dim}-d features -> Dense({NUM_CLASSES}); "
          f"params total {model.count_params():,}, trainable {feat_dim * NUM_CLASSES + NUM_CLASSES:,}")

    early_stop = EarlyStopping(monitor=CFG["MONITOR"], patience=CFG["PATIENCE"], restore_best_weights=True, verbose=1)
    timer = EpochTimer()
    start = time.perf_counter()
    history = model.fit(train_gen, epochs=CFG["MAX_EPOCHS"], validation_data=val_gen,
                        callbacks=[early_stop, timer], verbose=2)
    fit_time = time.perf_counter() - start

    hist = pd.DataFrame(history.history)
    hist.insert(0, "epoch", np.arange(1, len(hist) + 1))
    hist["epoch_time_s"] = timer.epoch_times

    # Make sure the best-epoch weights are the ones evaluated, and check them on the validation set
    best_idx = int(getattr(early_stop, "best_epoch", hist["val_loss"].to_numpy().argmin()))
    if early_stop.best_weights is not None:
        model.set_weights(early_stop.best_weights)
    val_loss_check, _ = model.evaluate(make_eval_generator(VAL_DIR, spec["size"], preprocess), verbose=0)
    restore_ok = bool(np.isclose(val_loss_check, hist["val_loss"].iloc[best_idx], rtol=1e-3, atol=1e-4))
    if not restore_ok:
        print(f"WARNING: restored val_loss {val_loss_check:.5f} != best-epoch val_loss {hist['val_loss'].iloc[best_idx]:.5f}")

    test_loss, test_acc = model.evaluate(test_gen, verbose=0)
    test_gen.reset()
    prob = model.predict(test_gen, verbose=0)
    y_true = test_gen.classes
    assert prob.shape == (test_gen.samples, NUM_CLASSES)
    if not np.isclose(test_acc, (prob.argmax(axis=1) == y_true).mean(), atol=1e-4):
        print("WARNING: Keras test accuracy differs from the prediction-based accuracy")

    if CFG["SAVE_WEIGHTS"]:
        model.save(out / "best.keras")

    info = common_info(name, "Keras", spec["family"], spec["size"], spec["pre"].__module__,
                       "top_dropout -> Dense(5)" if spec["head"] == "top" else "GlobalAveragePooling2D -> Dense(5)",
                       model.count_params(), feat_dim * NUM_CLASSES + NUM_CLASSES,
                       train_gen.samples, val_gen.samples, test_gen.samples)
    info.update(history_info(hist, best_idx, fit_time, early_stop.stopped_epoch > 0))
    info.update(test_loss=float(test_loss), keras_test_accuracy=float(test_acc),
                restore_check_val_loss=float(val_loss_check), restore_check_ok=restore_ok)

    res = finalize_run(name, info, hist, y_true, prob, test_gen.filenames)
    del model, early_stop, history
    cleanup_backend()
    return res

### 5.1 EfficientNetB0
224×224 · 1280-d features · student-sized baseline (not a teacher candidate)

In [ ]:
run_or_load("EfficientNetB0", run_keras_model);

### 5.2 EfficientNetB1
240×240 · 1280-d features

In [ ]:
run_or_load("EfficientNetB1", run_keras_model);

### 5.3 EfficientNetB2
260×260 · 1408-d features

In [ ]:
run_or_load("EfficientNetB2", run_keras_model);

### 5.4 EfficientNetB3
300×300 · 1536-d features

In [ ]:
run_or_load("EfficientNetB3", run_keras_model);

### 5.5 EfficientNetB4
380×380 · 1792-d features

In [ ]:
run_or_load("EfficientNetB4", run_keras_model);

### 5.6 EfficientNetB5
456×456 · 2048-d features

In [ ]:
run_or_load("EfficientNetB5", run_keras_model);

### 5.7 ResNet50
224×224 · 2048-d features · caffe preprocessing

In [ ]:
run_or_load("ResNet50", run_keras_model);

### 5.8 ResNet101
224×224 · 2048-d features · caffe preprocessing

In [ ]:
run_or_load("ResNet101", run_keras_model);

### 5.9 DenseNet121
224×224 · 1024-d features · torch preprocessing

In [ ]:
run_or_load("DenseNet121", run_keras_model);

### 5.10 DenseNet201
224×224 · 1920-d features · torch preprocessing

In [ ]:
run_or_load("DenseNet201", run_keras_model);

### 5.11 InceptionV3
299×299 · 2048-d features · tf preprocessing ([-1, 1])

In [ ]:
run_or_load("InceptionV3", run_keras_model);

### 5.12 InceptionResNetV2
299×299 · 1536-d features · tf preprocessing ([-1, 1])

In [ ]:
run_or_load("InceptionResNetV2", run_keras_model);

### 5.13 MobileNetV2
224×224 · 1280-d features · baseline (not a teacher candidate)

In [ ]:
run_or_load("MobileNetV2", run_keras_model);

## 6. PyTorch / timm Models (Twins, PVTv2)
### 6.0 Setup
Twins and PVTv2 are not available in Keras, so they use timm (PyTorch). To keep the setup identical, they read their batches from **the same Keras generators**, with the same BPHE, resize, augmentation and batch size. They follow the same recipe:
- frozen ImageNet backbone in inference mode, with timm's own pooling (token average for Twins, spatial average for PVTv2) → a new `Linear(512, 5)` head;
- softmax cross-entropy, Adam with learning rate 1e-4 (`eps=1e-7` as in Keras), batch 32, at most 100 epochs;
- early stopping with Keras's exact rule (`val_loss`, patience 10, restore best).

Normalisation: timm's ImageNet mean/std, applied through `densenet.preprocess_input` (`/255`, then mean/std), which gives identical values.

In [ ]:
IMAGENET_MEAN, IMAGENET_STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
TIMM_PRE = densenet.preprocess_input
TIMM_MODELS = {
    "Twins-SVT-S": dict(timm_name="twins_svt_small.in1k", size=224, family="Transformer"),
    "PVTv2-B2": dict(timm_name="pvt_v2_b2.in1k", size=224, family="Transformer"),
}
assert list(TIMM_MODELS) == TIMM_ORDER

# The Keras 'torch' preprocessing must equal (BPHE / 255 - mean) / std
_x = (np.random.RandomState(0).rand(64, 64, 3) * 255).astype(np.float32)
_expected = (bphe_uint8(_x.copy()) / 255.0 - np.array(IMAGENET_MEAN)) / np.array(IMAGENET_STD)
assert np.allclose(make_bphe(TIMM_PRE)(_x.copy()), _expected, atol=1e-5)


class KerasLikeEarlyStopping:
    """Same rule as keras.callbacks.EarlyStopping(monitor, patience, min_delta=0, restore_best_weights=True)."""

    def __init__(self, patience):
        self.patience, self.wait = patience, 0
        self.best, self.best_epoch, self.best_state, self.stopped_epoch = np.inf, 0, None, 0

    def step(self, epoch, value, module):
        """Call once per epoch (0-based). Returns True when training should stop."""
        if self.best_state is None:
            self.best_state = {k: v.detach().clone() for k, v in module.state_dict().items()}
        self.wait += 1
        if value < self.best:
            self.best, self.best_epoch, self.wait = value, epoch, 0
            self.best_state = {k: v.detach().clone() for k, v in module.state_dict().items()}
            return False
        if self.wait >= self.patience and epoch > 0:
            self.stopped_epoch = epoch
            return True
        return False


def build_timm_model(name):
    spec = TIMM_MODELS[name]
    model = timm.create_model(spec["timm_name"], pretrained=True, num_classes=NUM_CLASSES)
    data_cfg = timm.data.resolve_model_data_config(model)
    assert np.allclose(data_cfg["mean"], IMAGENET_MEAN) and np.allclose(data_cfg["std"], IMAGENET_STD)
    assert data_cfg["input_size"][-1] == spec["size"]

    model.requires_grad_(False)                  # freeze the backbone
    head = model.get_classifier()
    head.requires_grad_(True)                    # train only the new Linear head
    nn.init.xavier_uniform_(head.weight)         # same initialisation as a Keras Dense layer (glorot_uniform)
    nn.init.zeros_(head.bias)
    n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    assert n_trainable == model.num_features * NUM_CLASSES + NUM_CLASSES, n_trainable
    return model.to(DEVICE).eval(), head


def iter_torch_batches(gen):
    """Yield (NCHW float tensor, int label tensor) batches from a Keras DirectoryIterator."""
    for i in range(len(gen)):
        x, y = gen[i]
        yield (torch.from_numpy(np.ascontiguousarray(x.transpose(0, 3, 1, 2))).float().to(DEVICE),
               torch.from_numpy(y.argmax(axis=1)).long().to(DEVICE))


def train_timm_one_epoch(model, gen, optimizer, loss_fn):
    model.eval()                                 # frozen layers stay in inference mode, as in Keras
    total_loss, correct, seen = 0.0, 0, 0
    for xb, yb in iter_torch_batches(gen):
        logits = model(xb)
        loss = loss_fn(logits, yb)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * yb.size(0)
        correct += (logits.argmax(dim=1) == yb).sum().item()
        seen += yb.size(0)
    gen.on_epoch_end()                           # reshuffle for the next epoch, as Keras does
    return total_loss / seen, correct / seen


@torch.no_grad()
def predict_timm(model, gen, loss_fn):
    model.eval()
    probs, labels, total_loss, seen = [], [], 0.0, 0
    for xb, yb in iter_torch_batches(gen):
        logits = model(xb)
        total_loss += loss_fn(logits, yb).item() * yb.size(0)
        seen += yb.size(0)
        probs.append(torch.softmax(logits, dim=1).cpu().numpy())
        labels.append(yb.cpu().numpy())
    return np.concatenate(probs), np.concatenate(labels), total_loss / seen


def _backbone_checksum(model, head):
    head_ids = {id(p) for p in head.parameters()}
    return float(sum(p.detach().double().sum().item() for p in model.parameters() if id(p) not in head_ids))


def run_timm_model(name):
    spec = TIMM_MODELS[name]
    out = OUT_DIR / name
    out.mkdir(parents=True, exist_ok=True)
    keras.utils.set_random_seed(CFG["SEED"])
    torch.manual_seed(CFG["SEED"])

    preprocess = make_bphe(TIMM_PRE)
    train_gen, val_gen, test_gen = make_generators(spec["size"], preprocess, seed=CFG["SEED"])
    model, head = build_timm_model(name)
    n_params = sum(p.numel() for p in model.parameters())
    n_trainable = model.num_features * NUM_CLASSES + NUM_CLASSES
    print(f"{name} ({spec['timm_name']}): input {spec['size']}x{spec['size']}, {model.num_features}-d features -> "
          f"Linear({NUM_CLASSES}); params total {n_params:,}, trainable {n_trainable:,}; device {DEVICE}")
    checksum_before = _backbone_checksum(model, head)

    optimizer = torch.optim.Adam(head.parameters(), lr=CFG["LR"], eps=1e-7)
    loss_fn = nn.CrossEntropyLoss()
    early_stop = KerasLikeEarlyStopping(CFG["PATIENCE"])
    rows, start = [], time.perf_counter()
    for epoch in range(CFG["MAX_EPOCHS"]):
        t0 = time.perf_counter()
        train_loss, train_acc = train_timm_one_epoch(model, train_gen, optimizer, loss_fn)
        val_prob, val_y, val_loss = predict_timm(model, val_gen, loss_fn)
        val_gen.on_epoch_end()
        val_acc = float((val_prob.argmax(axis=1) == val_y).mean())
        dt = time.perf_counter() - t0
        rows.append(dict(epoch=epoch + 1, accuracy=train_acc, loss=train_loss, val_accuracy=val_acc,
                         val_loss=val_loss, epoch_time_s=dt))
        print(f"Epoch {epoch + 1}/{CFG['MAX_EPOCHS']} - {dt:.0f}s - accuracy: {train_acc:.4f} - loss: {train_loss:.4f}"
              f" - val_accuracy: {val_acc:.4f} - val_loss: {val_loss:.4f}")
        if early_stop.step(epoch, val_loss, head):
            print(f"Epoch {epoch + 1}: early stopping")
            break
    fit_time = time.perf_counter() - start
    hist = pd.DataFrame(rows)

    # Restore the best-epoch head and check it on the validation set
    head.load_state_dict(early_stop.best_state)
    print(f"Restoring model weights from the end of the best epoch: {early_stop.best_epoch + 1}.")
    _, _, val_loss_check = predict_timm(model, make_eval_generator(VAL_DIR, spec["size"], preprocess), loss_fn)
    restore_ok = bool(np.isclose(val_loss_check, hist["val_loss"].iloc[early_stop.best_epoch], rtol=1e-3, atol=1e-4))
    if not restore_ok:
        print(f"WARNING: restored val_loss {val_loss_check:.5f} != best-epoch val_loss "
              f"{hist['val_loss'].iloc[early_stop.best_epoch]:.5f}")
    backbone_unchanged = bool(np.isclose(_backbone_checksum(model, head), checksum_before, rtol=0, atol=1e-6))
    if not backbone_unchanged:
        print("WARNING: frozen backbone weights changed during training")

    prob, y_true, test_loss = predict_timm(model, test_gen, loss_fn)
    assert np.array_equal(y_true, test_gen.classes), "Test label order mismatch"

    if CFG["SAVE_WEIGHTS"]:
        torch.save({"timm_name": spec["timm_name"], "num_classes": NUM_CLASSES,
                    "head_state_dict": head.state_dict()}, out / "head_best.pt")

    info = common_info(name, "PyTorch (timm)", spec["family"], spec["size"], f"{TIMM_PRE.__module__} (ImageNet mean/std)",
                       "average pooling -> Linear(5)", n_params, n_trainable,
                       train_gen.samples, val_gen.samples, test_gen.samples)
    info.update(timm_name=spec["timm_name"])
    info.update(history_info(hist, early_stop.best_epoch, fit_time, early_stop.stopped_epoch > 0))
    info.update(test_loss=float(test_loss), restore_check_val_loss=float(val_loss_check),
                restore_check_ok=restore_ok, backbone_unchanged=backbone_unchanged)

    res = finalize_run(name, info, hist, y_true, prob, test_gen.filenames)
    del model, head, optimizer
    cleanup_backend()
    return res

### 6.1 Twins-SVT-S
`twins_svt_small.in1k` · 224×224 · 512-d features

In [ ]:
run_or_load("Twins-SVT-S", run_timm_model);

### 6.2 PVTv2-B2
`pvt_v2_b2.in1k` · 224×224 · 512-d features

In [ ]:
run_or_load("PVTv2-B2", run_timm_model);

## 7. Summary & Comparison
### 7.1 Results table
One row per finished model, read from `results/<ModelName>/metrics.json`, plus:
- **Custom-ENB5 (reference)**: notebook 3's printed results. It was trained with a different recipe (fine-tuned EfficientNetB5 with a 4-layer head, RMSprop 1e-4, 224 px, no BPHE, no early stopping, last-epoch weights), so compare it on test metrics only.
- `test_accuracy_6.1_run`: the test accuracy of the earlier 6.1 run of the same model (100 epochs, last-epoch weights), for reference. The 6.1 EfficientNetB0 and B5 runs read `/mnt/d/FA019/SplitDataset` rather than `new/SplittedDataset`, and the 6.1 runs saw 229–230 validation images.

In [ ]:
REFERENCE = dict(
    model="Custom-ENB5 (reference)", framework="Keras", family="EfficientNet", input_size=224,
    recipe="notebook 3: fine-tuned EfficientNetB5 + 4-layer head, RMSprop 1e-4, no early stopping",
    params_total=32_189_180, epochs_run=100, best_epoch=100, stopped_early=False,
    final_val_accuracy=0.8952, max_val_accuracy=0.9214, val_accuracy_at_best=0.8952,
    test_accuracy=0.8948, test_precision_weighted=0.8945, test_recall_weighted=0.8948, test_f1_weighted=0.8942,
    test_precision_macro=0.90, test_recall_macro=0.90, test_f1_macro=0.90, fit_time_min=96.2,
)
PREVIOUS_61_TEST_ACC = {"EfficientNetB0": 0.5987, "EfficientNetB4": 0.6459, "EfficientNetB5": 0.6223,
                        "ResNet50": 0.6009, "ResNet101": 0.6009, "DenseNet121": 0.5622, "DenseNet201": 0.6094,
                        "InceptionV3": 0.5687, "InceptionResNetV2": 0.6094}

finished = [m for m in ALL_MODELS if (OUT_DIR / m / "metrics.json").exists()]
missing = [m for m in ALL_MODELS if m not in finished]
print(f"Finished: {len(finished)}/{len(ALL_MODELS)}" + (f"  (missing: {', '.join(missing)})" if missing else ""))

records = [load_result(m, show=False) for m in finished]
COLUMNS = ["model", "framework", "family", "input_size", "params_total", "epochs_run", "best_epoch", "stopped_early",
           "best_val_loss", "val_accuracy_at_best", "max_val_accuracy", "test_accuracy", "test_precision_weighted",
           "test_recall_weighted", "test_f1_weighted", "test_precision_macro", "test_recall_macro", "test_f1_macro",
           "test_auc_macro", "test_auc_micro", "fit_time_min", "time_to_best_s", "epoch1_s", "median_epoch_s"]
summary = pd.DataFrame(records).reindex(columns=COLUMNS)
summary["time_to_best_min"] = summary.pop("time_to_best_s") / 60
summary["test_accuracy_6.1_run"] = summary["model"].map(PREVIOUS_61_TEST_ACC)
summary_all = pd.concat([summary, pd.DataFrame([REFERENCE]).reindex(columns=summary.columns)], ignore_index=True)
summary_all.to_csv(SUMMARY_DIR / "summary.csv", index=False)

advisor = summary_all[["model", "val_accuracy_at_best", "test_accuracy", "test_f1_macro", "fit_time_min"]].rename(
    columns={"model": "Model Name", "val_accuracy_at_best": "Validation Accuracy", "test_accuracy": "Test Accuracy",
             "test_f1_macro": "Test Macro F1", "fit_time_min": "Training Time (min)"})
advisor.to_csv(SUMMARY_DIR / "advisor_table.csv", index=False)

shown = summary_all.sort_values("test_f1_macro", ascending=False).reset_index(drop=True)
metric_cols = ["val_accuracy_at_best", "test_accuracy", "test_f1_weighted", "test_f1_macro", "test_auc_macro"]
try:
    display(shown.style.format(precision=4, na_rep="-").highlight_max(subset=metric_cols, color="#cfe8ff"))
except Exception:  # jinja2 not installed
    display(shown.round(4))
print("Saved:", SUMMARY_DIR / "summary.csv", "and", SUMMARY_DIR / "advisor_table.csv")

### 7.2 Comparison figures
Figures are saved to `results/_summary/figures/`:
1. test metrics per model;
2. training time, split into the first epoch (which includes XLA compilation for Keras) and the remaining epochs;
3. macro F1 against training time;
4. validation loss of every model.

In [ ]:
FIG_SUM = SUMMARY_DIR / "figures"
df = summary.copy()

if df.empty:
    print("No finished models yet.")
else:
    names, x = df["model"].tolist(), np.arange(len(df))

    # 1) Test metrics
    metrics = [("test_accuracy", "Accuracy"), ("test_f1_macro", "Macro F1"),
               ("test_f1_weighted", "Weighted F1"), ("test_auc_macro", "Macro AUC")]
    w = 0.8 / len(metrics)
    fig, ax = plt.subplots(figsize=(max(8.6, 1.05 * len(df)), 5.6))
    for i, (key, label) in enumerate(metrics):
        ax.bar(x + (i - (len(metrics) - 1) / 2) * w, df[key], w, label=label, color=OKABE_ITO[i],
               edgecolor="black", linewidth=1.0)
    ax.axhline(REFERENCE["test_accuracy"], color=C_REF, ls=":", lw=2.2, label="Custom-ENB5 test accuracy (ref.)")
    ax.set_xticks(x)
    ax.set_xticklabels(names, rotation=30, ha="right")
    ax.set_xlim(-0.6, len(df) - 0.4)
    ax.set_ylim(0, 1.15)
    ax.set_yticks(np.arange(0, 1.01, 0.2))
    ax.set_ylabel("Score (test set)")
    ax.set_title("Test-set performance")
    ax.grid(True, axis="y", alpha=0.3, linestyle="--")
    ax.set_axisbelow(True)
    ax.legend(ncol=5, fontsize=11, loc="upper center")
    save_fig(fig, FIG_SUM / "test_metrics")

    # 2) Training time
    first = df["epoch1_s"] / 60
    rest = df["fit_time_min"] - first
    fig, ax = plt.subplots(figsize=(max(8.6, 1.05 * len(df)), 5.6))
    ax.bar(x, first, 0.6, color=OKABE_ITO[4], edgecolor="black", linewidth=1.0, label="Epoch 1")
    ax.bar(x, rest, 0.6, bottom=first, color=OKABE_ITO[0], edgecolor="black", linewidth=1.0, label="Remaining epochs")
    ax.plot(x, df["time_to_best_min"], ls="none", marker="D", ms=8, color=C_VAL, label="Time to best epoch")
    for xi, total, n_ep in zip(x, df["fit_time_min"], df["epochs_run"]):
        ax.text(xi, total, f"{total:.1f}\n({n_ep} ep)", ha="center", va="bottom", fontsize=10, fontweight="bold")
    ax.set_xticks(x)
    ax.set_xticklabels(names, rotation=30, ha="right")
    ax.set_ylabel("Training time (min)")
    ax.set_title("Training time with early stopping")
    ax.grid(True, axis="y", alpha=0.3, linestyle="--")
    ax.set_axisbelow(True)
    ax.legend(fontsize=11, loc="upper left")
    save_fig(fig, FIG_SUM / "training_time")

    # 3) Macro F1 vs training time
    fig, ax = plt.subplots(figsize=(7.4, 5.6))
    sizes = 60 + 340 * df["params_total"] / df["params_total"].max()
    for fam, grp in df.groupby("family"):
        ax.scatter(grp["fit_time_min"], grp["test_f1_macro"], s=sizes[grp.index], color=FAMILY_COLORS.get(fam, "#7F7F7F"),
                   edgecolor="black", linewidth=1.0, label=fam, alpha=0.9)
    for _, r in df.iterrows():
        ax.annotate(r["model"], (r["fit_time_min"], r["test_f1_macro"]), textcoords="offset points", xytext=(6, 4),
                    fontsize=9, fontweight="bold")
    ax.set_xlabel("Training time (min)")
    ax.set_ylabel("Test macro F1")
    ax.set_title("Performance vs training cost\n(marker size = parameters)")
    ax.grid(True, alpha=0.3, linestyle="--")
    ax.set_axisbelow(True)
    ax.legend(fontsize=10, loc="best")
    save_fig(fig, FIG_SUM / "f1_vs_time")

    # 4) Validation loss of every model
    fig, ax = plt.subplots(figsize=(9.6, 6.0))
    styles = ["-", "--", "-.", ":"]
    fam_count = {}
    for _, r in df.iterrows():
        h = pd.read_csv(OUT_DIR / r["model"] / "history.csv")
        k = fam_count.get(r["family"], 0)
        fam_count[r["family"]] = k + 1
        ax.plot(h["epoch"], h["val_loss"], color=FAMILY_COLORS.get(r["family"], "#7F7F7F"), ls=styles[k % 4],
                lw=2.0, label=r["model"])
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Validation loss")
    ax.set_title("Validation loss")
    ax.grid(True, alpha=0.3, linestyle="--")
    ax.set_axisbelow(True)
    ax.legend(ncol=2, fontsize=9, loc="upper right")
    save_fig(fig, FIG_SUM / "val_loss_all")

### 7.3 Reading the results
- **Choosing a teacher.** Rank models by test macro F1 and accuracy (7.1). Differences of about 3 accuracy points or less (roughly 14 of the 466 test images) are within run-to-run noise.
- **Validation accuracy** in the tables is the value at the best epoch, i.e. the weights that were restored and tested.
- **Training time** covers `fit` only. Epoch 1 includes XLA compilation for the Keras models. PyTorch and Keras times are not directly comparable, because data loading is shared and CPU-bound.
- **Custom-ENB5** uses a different, fine-tuned recipe. It shows what full fine-tuning reaches, not a like-for-like comparison.
- The **best-epoch weights** of each model are saved (`best.keras` / `head_best.pt`) for the knowledge-distillation step.